## 2. Data Understanding & Cleaning

This notebook examines the structure, quality, and characteristics of the dataset used in the analysis and prepares it for exploratory analysis and predictive modeling.

## 2.1 Data Source

The dataset used in this project is the Diabetes 130-US Hospitals for Years 1999–2008 dataset from the UCI Machine Learning Repository. It contains 101,766 inpatient encounters collected from 130 U.S. hospitals and integrated delivery networks between 1999 and 2008. The dataset includes patient demographics, admission and discharge information, diagnoses, laboratory procedures, medications, and prior healthcare utilization.

Each row represents a hospital encounter, not a unique patient; therefore, a patient may appear in multiple records.

The `readmitted` variable classifies each encounter into three outcomes:

- `<30` — readmitted within 30 days
- `>30` — readmitted after 30 days
- `NO` — no recorded readmission

For this project, 30-day readmission (`<30`) is the outcome of interest.



## 2.2 Data Loading and Initial Inspection

The raw dataset is loaded without modification to examine its dimensions, structure, and data types.

### 2.2.1 Load the Dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/diabetic_data.csv")

df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### 2.2.2 Dataset Dimensions

In [3]:
# Check dataset dimensions
df.shape

(101766, 50)

The raw dataset contains 101,766 hospital encounters and 50 variables.

### 2.2.3 Column Overview

In [4]:
# Display column names
df.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

### 2.2.4 Data Types and Completeness


In [5]:
# Inspect data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

The dataset contains 13 integer variables and 37 categorical variables. Most columns appear complete based on standard null detection. 
Standard null counts alone are not sufficient to assess completeness. These values will be assessed separately during the data-quality assessment.

### 2.2.5 Unique Patients and Encounters

Because each row represents a hospital encounter, the number of unique patients is compared with the number of encounters to determine the extent to which patients appear multiple times in the dataset.

In [6]:
# Compare hospital encounters with unique patients
n_encounters = df["encounter_id"].nunique()
n_patients = df["patient_nbr"].nunique()

print(f"Unique encounters: {n_encounters:,}")
print(f"Unique patients: {n_patients:,}")
print(f"Repeat encounters: {n_encounters - n_patients:,}")

Unique encounters: 101,766
Unique patients: 71,518
Repeat encounters: 30,248


The dataset contains 101,766 unique hospital encounters involving 71,518 unique patients. This confirms that some patients appear in the dataset across multiple hospital encounters.

The difference of 30,248 between encounter count and unique patient count indicates repeated utilization within the study population. Because repeated encounters from the same patient are not independent, patient-level grouping will need to be considered when splitting the data for predictive modeling to reduce the risk of data leakage.

### 2.2.6 Target Variable Identification

The outcome variable for the analysis is `readmitted`, which records whether an encounter was followed by a hospital readmission within 30 days, after 30 days, or no recorded readmission.

For predictive modeling, the outcome will later be transformed into a binary target representing 30-day readmission:

- `<30` → 1
- `>30` → 0
- `NO` → 0

The transformation will be performed during feature engineering to preserve the original variable during data understanding and exploratory analysis.

## 2.3 Data Quality Assessment

Before cleaning the dataset, data quality is assessed to identify missing values, placeholder values, duplicate records, and other issues that may affect the analysis.

### 2.3.1 Missing Values

Standard null values are assessed across all variables to identify columns with incomplete observations.

In [7]:
# Assess standard missing values
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values("Missing_Percentage", ascending=False)

missing_summary

,Missing_Count,Missing_Percentage
max_glu_serum,96420,94.75
A1Cresult,84748,83.28


Two variables contain standard missing values: `max_glu_serum` (94.75%) and `A1Cresult` (83.28%). Their high missingness will be addressed during data cleaning.

### 2.3.2 Placeholder Missing Values

Categorical variables are examined for placeholder values that may represent missing or unknown information but are not recognized as null values by Pandas.

In [8]:
# Count '?' placeholder values in each column
placeholder_summary = pd.DataFrame({
    "Placeholder_Count": (df == "?").sum(),
    "Placeholder_Percentage": ((df == "?").mean() * 100).round(2)
})

placeholder_summary = placeholder_summary[
    placeholder_summary["Placeholder_Count"] > 0
].sort_values("Placeholder_Percentage", ascending=False)

placeholder_summary

,Placeholder_Count,Placeholder_Percentage
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


Seven variables contain `?` as a placeholder for missing or unknown information. Missingness is particularly high in `weight` (96.86%), `medical_specialty` (49.08%), and `payer_code` (39.56%). Lower levels of placeholder missingness are present in `race` (2.23%), `diag_3` (1.40%), `diag_2` (0.35%), and `diag_1` (0.02%).

These placeholder values are not recognized as standard null values and will therefore need to be accounted for during data cleaning.

### 2.3.3 Duplicate Records

The dataset is checked for exact duplicate rows.

In [9]:
# Check for exact duplicate rows
df.duplicated().sum()

np.int64(0)

No exact duplicate records were identified in the dataset.

### 2.3.4 Identifier Integrity

The encounter and patient identifiers are checked to confirm their expected uniqueness.

In [10]:
# Check identifier uniqueness
print(f"Duplicate encounter IDs: {df['encounter_id'].duplicated().sum():,}")
print(f"Duplicate patient IDs: {df['patient_nbr'].duplicated().sum():,}")

Duplicate encounter IDs: 0
Duplicate patient IDs: 30,248


All `encounter_id` values are unique, confirming that each row represents a distinct hospital encounter. A total of 30,248 `patient_nbr` values are repeated, consistent with patients having multiple encounters.

### 2.3.5 Categorical Value Consistency

Key categorical variables are examined for unexpected or inconsistent values.

In [11]:
# Inspect unique values in key categorical variables
categorical_cols = [
    "race",
    "gender",
    "age",
    "change",
    "diabetesMed",
    "readmitted"
]

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


race:
race
Caucasian          76099
AfricanAmerican    19210
?                   2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

gender:
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

age:
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

change:
change
No    54755
Ch    47011
Name: count, dtype: int64

diabetesMed:
diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64

readmitted:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


The categorical variables are generally consistent. `gender` contains 3 records classified as `Unknown/Invalid`, while the missing `race` values represented by `?` were identified previously.

### 2.3.6 Numerical Value Validation

Numerical variables are checked for invalid or implausible values before data cleaning.

In [12]:
# Define numerical variables to validate
numeric_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# Check minimum values for variables expected to be non-negative
df[numeric_cols].min()

time_in_hospital      1
num_lab_procedures    1
num_procedures        0
num_medications       1
number_outpatient     0
number_emergency      0
number_inpatient      0
number_diagnoses      1
dtype: int64

No invalid negative values were identified in the selected numerical variables.

### 2.3.7 Categorical Cardinality

The number of unique values in each categorical variable is examined to identify constant and high-cardinality features that may require further review.

In [13]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=["object", "string"]).columns

# Count unique values
categorical_cardinality = (
    df[categorical_cols]
    .nunique(dropna=False)
    .sort_values()
    .to_frame(name="Unique_Values")
)

categorical_cardinality

,Unique_Values
citoglipton,1
examide,1
acetohexamide,2
troglitazone,2
diabetesMed,2
glipizide-metformin,2
tolbutamide,2
metformin-rosiglitazone,2
metformin-pioglitazone,2
change,2


`examide` and `citoglipton` each contain only one unique value and therefore provide no variation for analysis.The diagnosis variables (`diag_1`, `diag_2`, and `diag_3`) have high cardinality, with more than 700 unique values each. These variables may require grouping or transformation before predictive modeling.

### 2.3.8 Coded Categorical Variables

Several variables are stored as integers but represent categorical codes rather than continuous numerical measurements. Their unique values are examined before determining how they should be handled in the analysis.

In [14]:
# Inspect coded categorical variables
coded_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in coded_cols:
    print(f"{col}: {sorted(df[col].unique())}")

admission_type_id: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
discharge_disposition_id: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(27), np.int64(28)]
admission_source_id: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(13), np.int64(14), np.int64(17), np.int64(20), np.int64(22), np.int64(25)]


`admission_type_id`, `discharge_disposition_id`, and `admission_source_id` are integer-coded categorical variables. Their numeric values represent categories rather than continuous quantities and will be treated accordingly during preprocessing.

## 2.4 Data Cleaning

The data-quality issues identified in the previous section are addressed to create a clean dataset for exploratory analysis and subsequent preprocessing.

### 2.4.1 Working Copy

A copy of the raw dataset is created so that cleaning operations do not modify the original data.

In [15]:
# Create a working copy
df_clean = df.copy()

### 2.4.2 Standardize Missing Values

Placeholder values represented by `?` are converted to standard missing values for consistent missing-data handling.

In [16]:
# Convert '?' placeholders to standard missing values
df_clean = df_clean.replace("?", np.nan)

Verify

In [17]:
# Confirm removal of '?' placeholders
(df_clean == "?").sum().sum()

np.int64(0)

In [18]:
# Reassess missingness after standardization
missing_clean = pd.DataFrame({
    "Missing_Count": df_clean.isna().sum(),
    "Missing_Percentage": (df_clean.isna().mean() * 100).round(2)
})

missing_clean = (
    missing_clean[missing_clean["Missing_Count"] > 0]
    .sort_values("Missing_Percentage", ascending=False)
)

missing_clean

,Missing_Count,Missing_Percentage
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


After standardizing `?` placeholders as missing values, nine variables contain missing data. Missingness ranges from 0.02% in `diag_1` to 96.86% in `weight`. The highest levels of missingness occur in `weight`, `max_glu_serum`, `A1Cresult`, `medical_specialty`, and `payer_code`.

These variables require individual treatment based on their missingness and analytical relevance.

### 2.4.3 Missing-Data Treatment

Variables with limited usable information or relevance are removed. Missing values in retained categorical variables are assigned explicit categories to preserve the affected encounters.

In [19]:
# Remove selected variables
df_clean.drop(
    columns=["weight", "payer_code"],
    inplace=True
)

# Preserve absence of recorded laboratory results
test_cols = ["max_glu_serum", "A1Cresult"]
df_clean[test_cols] = df_clean[test_cols].fillna("Not Tested")

# Preserve remaining encounters with missing categorical information
unknown_cols = ["medical_specialty", "race", "diag_1", "diag_2", "diag_3"]
df_clean[unknown_cols] = df_clean[unknown_cols].fillna("Unknown")

In [20]:
df_clean.isna().sum().sum()

np.int64(0)

All missing values have been addressed. The dataset now contains 48 variables after removing `weight` and `payer_code`.

### 2.4.4 Invalid and Non-Informative Values

Variables with no analytical variation and records containing invalid categorical values are addressed.

In [21]:
# Remove constant variables
df_clean.drop(columns=["examide", "citoglipton"], inplace=True)

# Remove invalid gender records
df_clean = df_clean[df_clean["gender"] != "Unknown/Invalid"].copy()

print(f"Dataset shape: {df_clean.shape}")
print(f"Invalid gender records: {(df_clean['gender'] == 'Unknown/Invalid').sum()}")

Dataset shape: (101763, 46)
Invalid gender records: 0


The three records with invalid gender values were removed, and the constant variables examide and citoglipton were dropped. The cleaned dataset now contains 101,763 encounters and 46 variables.

In [22]:
# Save cleaned dataset
df_clean.to_csv(
    "../data/processed/diabetic_data_cleaned.csv",
    index=False
)

### 2.4.5 Map Coded Variables

The coded admission, discharge, and admission-source variables are mapped to their corresponding descriptive categories using the dataset's ID mapping file.

In [23]:
# Load ID mapping file
id_map = pd.read_csv("../data/raw/IDS_mapping.csv", header=None)

# Extract mapping sections
admission_type_map = dict(zip(
    id_map.iloc[1:9, 0].dropna().astype(int),
    id_map.iloc[1:9, 1]
))

discharge_map = dict(zip(
    id_map.iloc[11:41, 0].dropna().astype(int),
    id_map.iloc[11:41, 1]
))

admission_source_map = dict(zip(
    id_map.iloc[43:68, 0].dropna().astype(int),
    id_map.iloc[43:68, 1]
))

# Create mapped dataset
df_mapped = df_clean.copy()

df_mapped["admission_type"] = df_mapped["admission_type_id"].map(admission_type_map)
df_mapped["discharge_disposition"] = df_mapped["discharge_disposition_id"].map(discharge_map)
df_mapped["admission_source"] = df_mapped["admission_source_id"].map(admission_source_map)

In [24]:
df_mapped[
    [
        "admission_type_id", "admission_type",
        "discharge_disposition_id", "discharge_disposition",
        "admission_source_id", "admission_source"
    ]
].head()

,admission_type_id,admission_type,discharge_disposition_id,discharge_disposition,admission_source_id,admission_source
0,6,NaN,25,Not Mapped,1,Physician Referral
1,1,Emergency,1,Discharged to home,7,Emergency Room
2,1,Emergency,1,Discharged to home,7,Emergency Room
3,1,Emergency,1,Discharged to home,7,Emergency Room
4,1,Emergency,1,Discharged to home,7,Emergency Room


The three coded administrative variables were mapped to descriptive categories using the provided ID mapping file, improving interpretability for subsequent analysis and dashboard development.

In [28]:
# Fill blank mapped descriptions
mapped_cols = [
    "admission_type",
    "discharge_disposition",
    "admission_source"
]

df_mapped[mapped_cols] = df_mapped[mapped_cols].fillna("Unknown")

# Verify
df_mapped[mapped_cols].isna().sum()

admission_type           0
discharge_disposition    0
admission_source         0
dtype: int64

In [29]:
# Final validation
print("Cleaned dataset shape:", df_clean.shape)
print("Mapped dataset shape:", df_mapped.shape)
print("Missing values:", df_mapped.isna().sum().sum())
print("'?' placeholders:", (df_mapped == "?").sum().sum())
print("Duplicate encounters:", df_mapped["encounter_id"].duplicated().sum())

Cleaned dataset shape: (101763, 46)
Mapped dataset shape: (101763, 49)
Missing values: 0
'?' placeholders: 0
Duplicate encounters: 0


All coded administrative variables now have descriptive values, with previously blank mappings classified as `Unknown/Not Mapped`.

In [30]:
df_clean.to_csv(
    "../data/processed/diabetic_data_cleaned.csv",
    index=False
)

df_mapped.to_csv(
    "../data/processed/diabetic_data_mapped.csv",
    index=False
)

The cleaned and mapped datasets are validated to confirm their dimensions, completeness, and consistency before exploratory analysis.

## 2.5 Summary

The data understanding and cleaning process established a reliable dataset for subsequent analysis.

Key outcomes include:

- The raw dataset contained 101,766 hospital encounters across 50 variables.
- Multiple encounters per patient were identified, confirming the need to account for repeated patients during predictive modeling.
- Missing and placeholder values were standardized and addressed without removing encounters due to missingness.
- `weight` and `payer_code` were removed due to high missingness and limited analytical relevance.
- `examide` and `citoglipton` were removed because they contained no variation.
- Three records with invalid gender values were removed.
- Coded admission, discharge, and admission-source variables were mapped to descriptive categories.
- The final mapped dataset contains 101,763 hospital encounters and is ready for exploratory data analysis.

In [ ]:
print("df_clean:")
print(df_clean["race"].value_counts(dropna=False))

print("\ndf_mapped:")
print(df_mapped["race"].value_counts(dropna=False))

df_clean:
race
Caucasian          76099
AfricanAmerican    19210
Unknown             2271
Hispanic            2037
Other               1505
Asian                641
Name: count, dtype: int64

df_mapped:
race
Caucasian          76099
AfricanAmerican    19210
Unknown             2271
Hispanic            2037
Other               1505
Asian                641
Name: count, dtype: int64
